In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import subprocess

# 나눔폰트 설치
subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'], check=True)

# 설치된 나눔폰트 파일 경로 자동 탐색 및 matplotlib 등록
font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
for fpath in font_files:
    fm.fontManager.addfont(fpath)

# 등록된 폰트 중 NanumBarunGothic 또는 NanumGothic 자동 선택
nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
selected_font = 'NanumBarunGothic' if 'NanumBarunGothic' in nanum_fonts else nanum_fonts[0]

plt.rc('font', family=selected_font)
plt.rcParams['axes.unicode_minus'] = False
print(f"적용된 한글 폰트: {selected_font}") 

적용된 한글 폰트: NanumBarunGothic


In [5]:
!pip install -q holidays

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import xgboost as xgb
import ipywidgets as widgets
import holidays
import csv
import subprocess
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output, HTML
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 자동 설치 및 적용
subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'], check=True)
for fpath in fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum']):
    fm.fontManager.addfont(fpath)
nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
selected_font = 'NanumBarunGothic' if 'NanumBarunGothic' in nanum_fonts else nanum_fonts[0]
plt.rc('font', family=selected_font)
plt.rcParams['axes.unicode_minus'] = False

# ==============================================================================
# 대한민국 설날·추석 명절 날짜 및 파생변수 생성 함수
# ==============================================================================
kr_holidays = holidays.KR(years=range(2015, 2030))
major_holiday_dates = sorted([
    pd.Timestamp(dt) for dt, name in kr_holidays.items()
    if any(k in name for k in ['설날', '추석', "New Year's Day", 'Seollal', 'Chuseok'])
    and not any(ex in name for ex in ['전날', '다음날', '대체', 'Holiday', 'Substitute', 'observed'])
])
if not major_holiday_dates:
    major_holiday_dates = sorted([
        pd.Timestamp(dt) for dt, name in kr_holidays.items()
        if any(k in name for k in ['설날', '추석', 'Seollal', 'Chuseok'])
    ])

major_holiday_arr = np.array(major_holiday_dates, dtype='datetime64[D]')
all_holiday_set = set(pd.to_datetime(list(kr_holidays.keys())))

def add_holiday_features(df_target, date_col):
    dates = pd.to_datetime(df_target[date_col]).values.astype('datetime64[D]')
    diff_matrix = (dates[:, None] - major_holiday_arr[None, :]).astype('timedelta64[D]').astype(int)
    nearest_idx = np.argmin(np.abs(diff_matrix), axis=1)
    d_day = diff_matrix[np.arange(len(dates)), nearest_idx]

    df_target['명절_D_day'] = np.where((d_day >= -21) & (d_day <= 10), d_day, 99)
    df_target['명절_성수기_여부'] = ((d_day >= -14) & (d_day <= -2)).astype(int)
    df_target['명절_직후_여부'] = ((d_day >= 1) & (d_day <= 5)).astype(int)
    df_target['명절_연휴_여부'] = pd.to_datetime(df_target[date_col]).isin(all_holiday_set).astype(int)
    return df_target

# ==============================================================================
# 손상된 행 복구 및 인코딩 오류 방지 데이터 로드
# ==============================================================================
FILE_PATH = '/content/kat_sale_citrus_merged.csv'
valid_rows, recovered_rows, dropped_count = [], [], 0

with open(FILE_PATH, 'r', encoding='utf-8-sig', errors='replace') as f:
    reader = csv.reader(f)
    header = []
    for row in reader:
        if any('trd_clcln_ymd' in str(col).lower() for col in row):
            header = [col.strip() for col in row]
            break

    expected_cols = len(header)
    for row in reader:
        if len(row) == expected_cols:
            valid_rows.append(row)
        else:
            recovered = False
            if len(row) > expected_cols:
                head_33 = row[:expected_cols]
                if head_33[1].strip().isdigit() and len(head_33[1].strip()) == 8 and len(head_33[-1].strip()) == 10 and '-' in head_33[-1]:
                    recovered_rows.append(head_33)
                    recovered = True
            if len(row) > 32:
                tail_32 = row[-32:]
                if tail_32[0].strip().isdigit() and len(tail_32[0].strip()) == 8 and len(tail_32[-1].strip()) == 10 and '-' in tail_32[-1]:
                    recovered_rows.append([''] + tail_32)
                    recovered = True
            if not recovered:
                dropped_count += 1

df = pd.DataFrame(valid_rows + recovered_rows, columns=header)
df.columns = df.columns.str.strip()

def get_col(name):
    for col in df.columns:
        if col.lower() == name.lower():
            return col
    return None

COL_DATE = get_col('Trd_clcln_ymd')
COL_VARIETY = get_col('Gds_sclsf_nm')
COL_GRADE = get_col('Grd_nm')
COL_SIZE = get_col('Sz_nm')
COL_VOLUME = get_col('Unit_tot_qty')
COL_TOT_AMT = get_col('Totprc')
COL_UNIT_QTY = get_col('Unit_qty')
COL_UNIT_NM = get_col('Unit_nm')
COL_MARKET = get_col('Query_whsl_mrkt_nm')
if COL_MARKET is None:
    cands = [c for c in df.columns if 'mrkt' in c.lower() and 'nm' in c.lower()]
    COL_MARKET = cands[0] if len(cands) > 0 else get_col('Whsl_mrkt_cd')

# 상자별(kg) 포장단위 컬럼 생성 (예: 3.0 + kg -> '3kg', 0.5 + kg -> '0.5kg')
COL_BOX = '포장단위'
use_cols = [COL_MARKET, COL_VARIETY, COL_GRADE, COL_SIZE, COL_DATE, COL_VOLUME, COL_TOT_AMT, COL_UNIT_QTY, COL_UNIT_NM]
data = df[use_cols].copy()
data = data[data[COL_DATE].astype(str) != COL_DATE]

data[COL_DATE] = pd.to_datetime(data[COL_DATE].astype(str), errors='coerce')
data[COL_VOLUME] = pd.to_numeric(data[COL_VOLUME].astype(str).str.replace(',', ''), errors='coerce')
data[COL_TOT_AMT] = pd.to_numeric(data[COL_TOT_AMT].astype(str).str.replace(',', ''), errors='coerce')
data[COL_UNIT_QTY] = pd.to_numeric(data[COL_UNIT_QTY].astype(str).str.replace(',', ''), errors='coerce')
data = data.dropna(subset=[COL_DATE, COL_VOLUME, COL_TOT_AMT, COL_UNIT_QTY])
data = data[data[COL_UNIT_QTY] > 0]

# kg 단위 데이터만 필터링 및 깔끔한 문자열 생성 (3.0 -> '3kg')
data[COL_UNIT_NM] = data[COL_UNIT_NM].astype(str).str.strip().str.lower()
data = data[data[COL_UNIT_NM] == 'kg']
qty_str = data[COL_UNIT_QTY].apply(lambda x: f"{int(x)}" if x.is_integer() else f"{x:g}")
data[COL_BOX] = qty_str + 'kg'

group_cols = [COL_MARKET, COL_VARIETY, COL_BOX, COL_GRADE, COL_SIZE]

for c in group_cols:
    data = data[~data[c].astype(str).str.contains('\ufffd', na=False)]
    data[c] = data[c].fillna('미분류').astype(str).str.strip()
    data[c] = data[c].replace({'': '미분류', '.': '미분류', 'nan': '미분류', 'None': '미분류'})

data = data[(data[COL_MARKET] != '미분류') & (~data[COL_VARIETY].isin(['미분류', '기타']))]
valid_df = data[(data[COL_VOLUME] > 0) & (data[COL_TOT_AMT] > 0)].copy()

agg_df = valid_df.groupby(group_cols + [COL_DATE], as_index=False).agg({COL_VOLUME: 'sum', COL_TOT_AMT: 'sum'})
agg_df['kg당_평균가격'] = agg_df[COL_TOT_AMT] / agg_df[COL_VOLUME]
agg_df = agg_df.sort_values(by=group_cols + [COL_DATE]).reset_index(drop=True)
print(f"1단계 완료! 상자 규격별 일별 집계 데이터: {len(agg_df):,}건 (주요 상자: {agg_df[COL_BOX].value_counts().head(5).to_dict()})")

agg_df['거래공백일수'] = agg_df.groupby(group_cols)[COL_DATE].diff().dt.days.fillna(1)
agg_df['시즌번호'] = (agg_df['거래공백일수'] >= 21).groupby([agg_df[c] for c in group_cols]).cumsum()
agg_df = agg_df[agg_df.groupby(group_cols + ['시즌번호'])[COL_DATE].transform('count') >= 3].reset_index(drop=True)

s_grp = group_cols + ['시즌번호']
agg_df['직전_거래량'] = agg_df.groupby(s_grp)[COL_VOLUME].shift(1)
agg_df['최근3일_평균거래량'] = agg_df.groupby(s_grp)[COL_VOLUME].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
agg_df['최근7일_평균거래량'] = agg_df.groupby(s_grp)[COL_VOLUME].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
agg_df['최근7일_거래량표준편차'] = agg_df.groupby(s_grp)[COL_VOLUME].transform(lambda x: x.shift(1).rolling(7, min_periods=1).std()).fillna(0)

agg_df['직전_가격'] = agg_df.groupby(s_grp)['kg당_평균가격'].shift(1)
agg_df['최근3일_평균가격'] = agg_df.groupby(s_grp)['kg당_평균가격'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
agg_df['최근7일_평균가격'] = agg_df.groupby(s_grp)['kg당_평균가격'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
agg_df = agg_df.dropna(subset=['직전_거래량', '직전_가격']).reset_index(drop=True)

def remove_iqr(g):
    if len(g) < 15:
        return g
    for col in [COL_VOLUME, 'kg당_평균가격']:
        Q1, Q3 = g[col].quantile(0.25), g[col].quantile(0.75)
        IQR = Q3 - Q1
        g = g[(g[col] >= Q1 - 3.0 * IQR) & (g[col] <= Q3 + 3.0 * IQR)]
    return g

df_norm = agg_df.groupby(group_cols, group_keys=False).apply(remove_iqr).reset_index(drop=True)
print(f"2단계 완료! 유효 데이터: {len(df_norm):,}건 (최신일: {df_norm[COL_DATE].max().strftime('%Y-%m-%d')})")

s_grp = group_cols + ['시즌번호']
df_norm = df_norm.sort_values(by=s_grp + [COL_DATE]).reset_index(drop=True)
df_norm['월'] = df_norm[COL_DATE].dt.month
df_norm['요일'] = df_norm[COL_DATE].dt.dayofweek
df_norm['주차'] = df_norm[COL_DATE].dt.isocalendar().week.astype(int)
df_norm['연중일수_sin'] = np.sin(2 * np.pi * df_norm[COL_DATE].dt.dayofyear / 365.0)
df_norm['연중일수_cos'] = np.cos(2 * np.pi * df_norm[COL_DATE].dt.dayofyear / 365.0)
df_norm['시즌_경과일수'] = df_norm.groupby(s_grp).cumcount() + 1
df_norm['휴장직후_개장일'] = (df_norm['거래공백일수'] >= 2).astype(int)

# 명절 변수 4종 추가
df_norm = add_holiday_features(df_norm, COL_DATE)

df_norm['순번'] = df_norm.groupby(group_cols).cumcount()
df_norm['전체수'] = df_norm.groupby(group_cols)[COL_DATE].transform('count')
is_train = df_norm['순번'] < (df_norm['전체수'] * 0.8)

# 학습 구간 기준 [시장·품종·포장단위·등급·사이즈·요일] 중앙값 계산
t_stats = df_norm[is_train].groupby(group_cols + ['요일'], as_index=False).agg(
    요일_기준거래량=(COL_VOLUME, 'median'),
    요일_기준단가=('kg당_평균가격', 'median')
)
df_norm = df_norm.merge(t_stats, on=group_cols + ['요일'], how='left')
df_norm['요일_기준거래량'] = df_norm['요일_기준거래량'].fillna(df_norm['최근7일_평균거래량'])
df_norm['요일_기준단가'] = df_norm['요일_기준단가'].fillna(df_norm['최근7일_평균가격'])

# 거래량 고도화 피처
df_norm['기준거래량_log'] = np.log1p(df_norm['요일_기준거래량'])
df_norm['직전_거래량_log'] = np.log1p(df_norm['직전_거래량'])
df_norm['최근3일_거래량_log'] = np.log1p(df_norm['최근3일_평균거래량'])
df_norm['최근7일_거래량_log'] = np.log1p(df_norm['최근7일_평균거래량'])

df_norm['직전_대비_기준거래량비'] = (df_norm['직전_거래량'] + 1) / (df_norm['요일_기준거래량'] + 1)
df_norm['최근3일_대비_기준거래량비'] = (df_norm['최근3일_평균거래량'] + 1) / (df_norm['요일_기준거래량'] + 1)
df_norm['최근7일_대비_기준거래량비'] = (df_norm['최근7일_평균거래량'] + 1) / (df_norm['요일_기준거래량'] + 1)
df_norm['거래량_변동계수'] = df_norm['최근7일_거래량표준편차'] / (df_norm['최근7일_평균거래량'] + 1)

# 단가 고도화 피처
df_norm['직전_단가_배율'] = df_norm['직전_가격'] / (df_norm['최근7일_평균가격'] + 1)
df_norm['최근3일_단가_배율'] = df_norm['최근3일_평균가격'] / (df_norm['최근7일_평균가격'] + 1)
df_norm['요일_단가_배율'] = df_norm['요일_기준단가'] / (df_norm['최근7일_평균가격'] + 1)

df_norm['target_vol_log_diff'] = (np.log1p(df_norm[COL_VOLUME]) - df_norm['기준거래량_log']).clip(-2.0, 2.0)
df_norm['target_price_ratio'] = (df_norm['kg당_평균가격'] / (df_norm['최근7일_평균가격'] + 1)).clip(0.3, 2.5)

for c in group_cols:
    df_norm[c] = df_norm[c].astype('category')

vol_features = [
    COL_MARKET, COL_VARIETY, COL_BOX, COL_GRADE, COL_SIZE, '월', '요일', '주차', '연중일수_sin', '연중일수_cos',
    '거래공백일수', '휴장직후_개장일', '시즌_경과일수', '명절_D_day', '명절_성수기_여부', '명절_직후_여부', '명절_연휴_여부',
    '기준거래량_log', '직전_거래량_log', '최근3일_거래량_log', '최근7일_거래량_log',
    '직전_대비_기준거래량비', '최근3일_대비_기준거래량비', '최근7일_대비_기준거래량비', '거래량_변동계수'
]

price_features = [
    COL_MARKET, COL_VARIETY, COL_BOX, COL_GRADE, COL_SIZE, '월', '요일', '주차', '연중일수_sin', '연중일수_cos',
    '거래공백일수', '휴장직후_개장일', '시즌_경과일수', '명절_D_day', '명절_성수기_여부', '명절_직후_여부', '명절_연휴_여부',
    '직전_대비_기준거래량비', '거래량_변동계수', '직전_단가_배율', '최근3일_단가_배율', '요일_단가_배율'
]

train_n = df_norm[is_train].dropna(subset=vol_features + price_features + ['target_vol_log_diff', 'target_price_ratio']).copy()
test_n = df_norm[~is_train].dropna(subset=vol_features + price_features + ['target_vol_log_diff', 'target_price_ratio']).copy()

m_vol_model = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=500, learning_rate=0.03, max_depth=5, min_child_weight=15,
    subsample=0.8, colsample_bytree=0.8, enable_categorical=True, random_state=42
)
m_vol_model.fit(train_n[vol_features], train_n['target_vol_log_diff'])
test_n['예측_거래량'] = np.maximum(np.expm1(test_n['기준거래량_log'] + m_vol_model.predict(test_n[vol_features])), 0)

m_price_ratio = xgb.XGBRegressor(
    n_estimators=400, learning_rate=0.03, max_depth=4, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.8, enable_categorical=True, random_state=42
)
m_price_ratio.fit(train_n[price_features], train_n['target_price_ratio'])
test_n['예측_단가'] = np.maximum(m_price_ratio.predict(test_n[price_features]) * test_n['최근7일_평균가격'], 0)

print(f"3단계 완료! 학습: {len(train_n):,}건 | 검증: {len(test_n):,}건")

def show_pretty_table(df_target, title_text, format_dict, gradient_cols=None):
    display(HTML(f"<h3 style='color:#1f4e78; margin-top:15px;'>{title_text}</h3>"))
    st = df_target.style.format(format_dict).set_properties(**{'text-align': 'center', 'padding': '7px 12px', 'border': '1px solid #ddd'}).hide(axis='index')
    if gradient_cols:
        st = st.background_gradient(subset=gradient_cols, cmap='Blues')
    display(st)

summary_rows = []
for keys, sub in test_n.groupby(group_cols, observed=True):
    if len(sub) < 5: continue
    mkt, var, box, grd, siz = keys
    y_v, p_v = sub[COL_VOLUME].values, sub['예측_거래량'].values
    y_p, p_p = sub['kg당_평균가격'].values, sub['예측_단가'].values
    vol_acc = max(0.0, 100.0 - (np.sum(np.abs(y_v - p_v)) / np.sum(y_v) * 100.0))
    price_acc = max(0.0, 100.0 - (np.sum(np.abs(y_p - p_p)) / np.sum(y_p) * 100.0))
    summary_rows.append({'도매시장': mkt, '품종': var, '포장단위': box, '등급': grd, '사이즈': siz, '검증일수': len(sub), '단가_정확도': price_acc, '단가_MAE': mean_absolute_error(y_p, p_p), '거래량_정확도': vol_acc, '거래량_MAE': mean_absolute_error(y_v, p_v)})

summary_df = pd.DataFrame(summary_rows).sort_values(by=['단가_정확도', '거래량_정확도'], ascending=False).reset_index(drop=True)
summary_df.to_csv('/content/citrus_total_accuracy_list.csv', index=False, encoding='utf-8-sig')
show_pretty_table(summary_df.head(30), "📊 전국 도매시장·품종·상자(kg)·등급·사이즈별 종합 정확도 (상위 30개)", {'검증일수': '{:,}일', '단가_정확도': '{:.2f}%', '단가_MAE': '{:,.0f} 원', '거래량_정확도': '{:.2f}%', '거래량_MAE': '{:,.1f} kg'}, ['단가_정확도', '거래량_정확도'])

today_str = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d')
future_dates = pd.date_range(start=df_norm[COL_DATE].min(), end=pd.to_datetime(today_str) + timedelta(days=30)).strftime('%Y-%m-%d').tolist()

markets = sorted(df_norm[COL_MARKET].astype(str).unique().tolist())
w_mkt = widgets.Dropdown(options=markets, value='서울가락' if '서울가락' in markets else markets[0], description='도매시장:')
w_var = widgets.Dropdown(description='감귤품종:')
w_box = widgets.Dropdown(description='포장(kg):')
w_grd = widgets.Dropdown(description='등급:')
w_siz = widgets.Dropdown(description='사이즈:')
w_date = widgets.Dropdown(options=future_dates, value=today_str if today_str in future_dates else future_dates[-1], description='기준날짜:')
w_days = widgets.Dropdown(options=[('향후 30일 (1개월)', 30), ('향후 15일 (보름)', 15), ('향후 7일 (1주)', 7)], value=30, description='예측기간:')

def update_var(*a):
    opts = sorted(df_norm[df_norm[COL_MARKET].astype(str) == w_mkt.value][COL_VARIETY].astype(str).unique().tolist())
    w_var.options = opts
    if opts: w_var.value = '조생귤' if '조생귤' in opts else opts[0]

def update_box(*a):
    opts = df_norm[(df_norm[COL_MARKET].astype(str) == w_mkt.value) & (df_norm[COL_VARIETY].astype(str) == w_var.value)][COL_BOX].astype(str).unique().tolist()
    opts = sorted(opts, key=lambda x: float(x.replace('kg', '')))
    w_box.options = opts
    if opts: w_box.value = '5kg' if '5kg' in opts else ('3kg' if '3kg' in opts else opts[0])

def update_grd(*a):
    opts = sorted(df_norm[(df_norm[COL_MARKET].astype(str) == w_mkt.value) & (df_norm[COL_VARIETY].astype(str) == w_var.value) & (df_norm[COL_BOX].astype(str) == w_box.value)][COL_GRADE].astype(str).unique().tolist())
    w_grd.options = opts
    if opts: w_grd.value = '특' if '특' in opts else opts[0]

def update_siz(*a):
    opts = sorted(df_norm[(df_norm[COL_MARKET].astype(str) == w_mkt.value) & (df_norm[COL_VARIETY].astype(str) == w_var.value) & (df_norm[COL_BOX].astype(str) == w_box.value) & (df_norm[COL_GRADE].astype(str) == w_grd.value)][COL_SIZE].astype(str).unique().tolist())
    w_siz.options = opts
    if opts: w_siz.value = opts[0]

w_mkt.observe(update_var, 'value')
w_var.observe(update_box, 'value')
w_box.observe(update_grd, 'value')
w_grd.observe(update_siz, 'value')
update_var(); update_box(); update_grd(); update_siz()
print(f"4단계 완료! 실시간 기준일: {today_str} (최대 예측 가능일: {future_dates[-1]})")

def render_test_results(mkt, var, box, grd, siz):
    box_kg = float(box.replace('kg', ''))
    sub = test_n[
        (test_n[COL_MARKET].astype(str) == mkt) &
        (test_n[COL_VARIETY].astype(str) == var) &
        (test_n[COL_BOX].astype(str) == box) &
        (test_n[COL_GRADE].astype(str) == grd) &
        (test_n[COL_SIZE].astype(str) == siz)
    ].sort_values(COL_DATE).copy()

    display(HTML("<hr style='margin:25px 0; border:1px solid #ccc;'>"))
    if len(sub) < 3:
        display(HTML(f"<div style='background-color:#f8f9fa; border-left:5px solid #6c757d; padding:12px;'>ℹ️ <b>[{mkt} > {var} > {box} > {grd} > {siz}]</b> 조합은 개별 테스트 표본이 적어({len(sub)}일) 통합 모델 가중치로 미래를 예측했습니다.</div>"))
        return

    y_p, p_p = sub['kg당_평균가격'].values, sub['예측_단가'].values
    y_v, p_v = sub[COL_VOLUME].values, sub['예측_거래량'].values
    p_acc = max(0.0, 100.0 - (np.sum(np.abs(y_p - p_p)) / np.sum(y_p) * 100.0))
    v_acc = max(0.0, 100.0 - (np.sum(np.abs(y_v - p_v)) / np.sum(y_v) * 100.0))

    x_labels = sub[COL_DATE].dt.strftime('%Y-%m-%d').values
    x_idx = np.arange(len(x_labels))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True)

    ax1.plot(x_idx, y_p * box_kg, label=f'실제 {box} 상자 단가 (원)', color='#2ca02c', marker='o', markersize=4)
    ax1.plot(x_idx, p_p * box_kg, label=f'예측 {box} 상자 단가 (원)', color='#d62728', linestyle='--', marker='x', markersize=4)
    ax1.set_title(f"[테스트 검증] {mkt} | {var} | {box} | {grd} | {siz} — 상자 단가 비교 (정확도: {p_acc:.2f}%)", fontweight='bold')
    ax1.set_ylabel(f'{box} 상자당 단가 (원)'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(x_idx, y_v, label='실제 거래량 (kg)', color='#1f77b4', marker='o', markersize=4)
    ax2.plot(x_idx, p_v, label='예측 거래량 (kg)', color='#ff7f0e', linestyle='--', marker='x', markersize=4)
    ax2.set_title(f"[테스트 검증] {mkt} | {var} | {box} | {grd} | {siz} — 거래량 비교 (정확도: {v_acc:.2f}%)", fontweight='bold')
    ax2.set_ylabel('거래량 (kg)'); ax2.legend(); ax2.grid(True, alpha=0.3)

    step = max(1, len(x_labels) // 15)
    plt.xticks(x_idx[::step], x_labels[::step], rotation=45, ha='right')
    plt.tight_layout(); plt.show()

    test_table = pd.DataFrame({
        '테스트일자': x_labels,
        '실제_1kg단가': y_p,
        '예측_1kg단가': p_p,
        f'실제_{box}상자가': y_p * box_kg,
        f'예측_{box}상자가': p_p * box_kg,
        '상자가_오차': (p_p - y_p) * box_kg,
        '실제_거래량': y_v,
        '예측_거래량': p_v
    }).tail(15)

    show_pretty_table(
        test_table,
        f"🔍 과거 20% 테스트 검증 상세 비교표 ({box} 기준 / 최근 {len(test_table)}영업일)",
        {
            '실제_1kg단가': '{:,.0f} 원', '예측_1kg단가': '{:,.0f} 원',
            f'실제_{box}상자가': '{:,.0f} 원', f'예측_{box}상자가': '{:,.0f} 원',
            '상자가_오차': '{:+,.0f} 원',
            '실제_거래량': '{:,.0f} kg', '예측_거래량': '{:,.0f} kg'
        }
    )

print("5단계 완료! 이어서 6단계를 실행하세요.")

btn_forecast = widgets.Button(description='미래 예측 및 테스트 검증 결과 조회', button_style='primary', icon='calendar', layout=widgets.Layout(width='280px'))
out_box = widgets.Output()

def run_all_views(b):
    with out_box:
        clear_output(wait=True)
        mkt, var, box, grd, siz = w_mkt.value, w_var.value, w_box.value, w_grd.value, w_siz.value
        box_kg = float(box.replace('kg', ''))
        base_date_str, horizon = w_date.value, w_days.value
        base_dt = pd.to_datetime(base_date_str)

        cond = (
            (df_norm[COL_MARKET].astype(str) == mkt) &
            (df_norm[COL_VARIETY].astype(str) == var) &
            (df_norm[COL_BOX].astype(str) == box) &
            (df_norm[COL_GRADE].astype(str) == grd) &
            (df_norm[COL_SIZE].astype(str) == siz)
        )
        hist_df = df_norm[cond & (df_norm[COL_DATE] <= base_dt)].sort_values(COL_DATE).copy()
        actual_future_df = df_norm[cond & (df_norm[COL_DATE] > base_dt) & (df_norm[COL_DATE] <= base_dt + timedelta(days=horizon))].sort_values(COL_DATE)

        target_months = {(base_dt + timedelta(days=i)).month for i in range(1, horizon + 1)}
        has_season_hist = df_norm[cond & (df_norm[COL_DATE].dt.month.isin(target_months))]

        if len(hist_df) < 7 or len(has_season_hist) < 3:
            display(HTML(f"<div style='background-color:#fff3cd; border-left:5px solid #ffc107; padding:12px; margin-top:10px; font-size:14px;'>⚠️ <b>데이터가 부족해서 미래 예측이 안 됩니다.</b><br>선택하신 조건(<b>{mkt} > {var} > {box} > {grd} > {siz}</b>)은 <b>{base_date_str}</b> 이후 출하 시즌 데이터가 부족합니다.</div>"))
            render_test_results(mkt, var, box, grd, siz)
            return

        acc_row = summary_df[(summary_df['도매시장'] == mkt) & (summary_df['품종'] == var) & (summary_df['포장단위'] == box) & (summary_df['등급'] == grd) & (summary_df['사이즈'] == siz)]
        if not acc_row.empty:
            p_val = float(next(iter(acc_row['단가_정확도'])))
            v_val = float(next(iter(acc_row['거래량_정확도'])))
            acc_text = f"검증 정확도 — 단가: <b>{p_val:.2f}%</b> | 거래량: <b>{v_val:.2f}%</b>"
        else:
            acc_text = "통합 모델 가중치 적용"

        vol_buf, prc_buf = hist_df[COL_VOLUME].tail(7).tolist(), hist_df['kg당_평균가격'].tail(7).tolist()
        season_day, last_trade_dt = int(hist_df['시즌_경과일수'].iloc[-1]), hist_df[COL_DATE].max()
        yoil_names, forecast_rows = ['월', '화', '수', '목', '금', '토', '일'], []

        box_col_name = f"{box}_상자예상가"
        act_box_col_name = f"{box}_실제상자가"

        for step in range(1, horizon + 1):
            target_dt = base_dt + timedelta(days=step)
            yoil = target_dt.dayofweek
            d_label = f"{target_dt.strftime('%Y-%m-%d')} ({yoil_names[yoil]})"
            if yoil == 6:
                forecast_rows.append({'예측일자': d_label, '주차': f"{(step-1)//7 + 1}주차", '시장상태': '일요 휴무', '예측_1kg단가': np.nan, box_col_name: np.nan, act_box_col_name: np.nan, '예측_거래량(kg)': np.nan, '예측_반입상자수': np.nan})
                continue

            season_day += 1
            gap_days = min((target_dt - last_trade_dt).days, 7)
            v_ma3, p_ma3, v_ma7, p_ma7 = np.mean(vol_buf[-3:]), np.mean(prc_buf[-3:]), np.mean(vol_buf[-7:]), np.mean(prc_buf[-7:])
            v_std7 = np.std(vol_buf[-7:], ddof=1) if len(vol_buf) > 1 else 0.0
            st_row = t_stats[(t_stats[COL_MARKET] == mkt) & (t_stats[COL_VARIETY] == var) & (t_stats[COL_BOX] == box) & (t_stats[COL_GRADE] == grd) & (t_stats[COL_SIZE] == siz) & (t_stats['요일'] == yoil)]
            v_yb = float(next(iter(st_row['요일_기준거래량']))) if not st_row.empty else v_ma7
            p_yb = float(next(iter(st_row['요일_기준단가']))) if not st_row.empty else p_ma7

            base_vol_log = np.log1p(v_yb)
            row_df = pd.DataFrame([{
                'target_dt': target_dt,
                COL_MARKET: mkt, COL_VARIETY: var, COL_BOX: box, COL_GRADE: grd, COL_SIZE: siz, '월': target_dt.month, '요일': yoil, '주차': int(target_dt.isocalendar().week),
                '연중일수_sin': np.sin(2 * np.pi * target_dt.dayofyear / 365.0), '연중일수_cos': np.cos(2 * np.pi * target_dt.dayofyear / 365.0),
                '거래공백일수': gap_days, '휴장직후_개장일': int(gap_days >= 2), '시즌_경과일수': season_day,
                '기준거래량_log': base_vol_log, '직전_거래량_log': np.log1p(vol_buf[-1]),
                '최근3일_거래량_log': np.log1p(v_ma3), '최근7일_거래량_log': np.log1p(v_ma7),
                '직전_대비_기준거래량비': (vol_buf[-1] + 1) / (v_yb + 1),
                '최근3일_대비_기준거래량비': (v_ma3 + 1) / (v_yb + 1),
                '최근7일_대비_기준거래량비': (v_ma7 + 1) / (v_yb + 1),
                '거래량_변동계수': v_std7 / (v_ma7 + 1),
                '직전_단가_배율': prc_buf[-1] / (p_ma7 + 1),
                '최근3일_단가_배율': p_ma3 / (p_ma7 + 1),
                '요일_단가_배율': p_yb / (p_ma7 + 1)
            }])
            row_df = add_holiday_features(row_df, 'target_dt')

            if row_df['명절_연휴_여부'].iloc[0] == 1:
                mkt_status = '명절/공휴일'
            elif row_df['명절_성수기_여부'].iloc[0] == 1:
                mkt_status = f"명절성수기(D{row_df['명절_D_day'].iloc[0]})"
            elif row_df['명절_직후_여부'].iloc[0] == 1:
                mkt_status = f"명절직후(D+{row_df['명절_D_day'].iloc[0]})"
            else:
                mkt_status = '정상 경매'

            for c in group_cols: row_df[c] = row_df[c].astype('category')
            pred_v = max(float(np.expm1(base_vol_log + m_vol_model.predict(row_df[vol_features]).item())), 0.0)
            pred_p = max(float(m_price_ratio.predict(row_df[price_features]).item()) * p_ma7, 0.0)
            act_m = actual_future_df[actual_future_df[COL_DATE] == target_dt]
            act_box_p = float(next(iter(act_m['kg당_평균가격']))) * box_kg if not act_m.empty else np.nan

            forecast_rows.append({
                '예측일자': d_label,
                '주차': f"{(step-1)//7 + 1}주차",
                '시장상태': mkt_status,
                '예측_1kg단가': pred_p,
                box_col_name: pred_p * box_kg,
                act_box_col_name: act_box_p,
                '예측_거래량(kg)': pred_v,
                '예측_반입상자수': pred_v / box_kg
            })
            vol_buf.append(pred_v); prc_buf.append(pred_p); last_trade_dt = target_dt

        res_df = pd.DataFrame(forecast_rows)
        valid_days = res_df[res_df['시장상태'] != '일요 휴무']
        best_day = valid_days.loc[valid_days[box_col_name].idxmax()]
        bp_box = best_day[box_col_name]
        bp_1kg = best_day['예측_1kg단가']
        wk_str = " | ".join([f"{k}: <b>{v:,.0f}원</b>" for k, v in valid_days.groupby('주차')[box_col_name].mean().items()])

        display(HTML(f"<div style='background-color:#eef5fc; border-left:5px solid #1f4e78; padding:12px; margin-top:10px; line-height:1.6;'><b>📅 향후 {horizon}일 미래 시세 예측 ({mkt} > {var} > {box} > {grd}등급 > {siz})</b> | {acc_text}<br>• 최고가 예상일: <b style='color:#d9534f;'>{best_day['예측일자']}</b> ({box} 상자: <b>{bp_box:,.0f}원</b> / 1kg 환산: <b>{bp_1kg:,.0f}원</b>)<br>• 주차별 {box} 상자 평균 예상가: {wk_str}</div>"))
        show_pretty_table(
            res_df,
            f"📦 [미래 예측] 향후 {horizon}일 {box} 상자 일별 예상 경락가",
            {
                '예측_1kg단가': lambda x: '-' if pd.isna(x) else f"{x:,.0f} 원",
                box_col_name: lambda x: '-' if pd.isna(x) else f"{x:,.0f} 원",
                act_box_col_name: lambda x: '미래예측' if pd.isna(x) else f"{x:,.0f} 원",
                '예측_거래량(kg)': lambda x: '-' if pd.isna(x) else f"{x:,.0f} kg",
                '예측_반입상자수': lambda x: '-' if pd.isna(x) else f"{x:,.0f} 상자"
            },
            [box_col_name]
        )

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(valid_days['예측일자'], valid_days[box_col_name], label=f'{box} 상자 예상가 (원)', color='#d62728', marker='o', linewidth=2)
        ax.set_title(f"[미래 예측] {mkt} | {var} | {box} | {grd} | {siz} — 향후 {horizon}일 {box} 상자 예상가 추이", fontweight='bold')
        ax.set_ylabel(f'{box} 상자당 예상가 (원)'); plt.xticks(rotation=45, ha='right'); ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

        render_test_results(mkt, var, box, grd, siz)

btn_forecast.on_click(run_all_views)
display(HTML("<h3 style='color:#1f4e78; margin-top:15px;'>🎛️ 실시간 감귤 상자별(kg) 미래 시세 예측 대시보드</h3>"))
display(widgets.VBox([widgets.HBox([w_mkt, w_var, w_box]), widgets.HBox([w_grd, w_siz, w_date, w_days, btn_forecast])]), out_box)
run_all_views(None)


1단계 완료! 상자 규격별 일별 집계 데이터: 692,692건 (주요 상자: {'5kg': 294547, '3kg': 211920, '9kg': 62866, '10kg': 57153, '0.5kg': 28771})
2단계 완료! 유효 데이터: 609,342건 (최신일: 2026-11-11)
3단계 완료! 학습: 491,879건 | 검증: 117,463건


도매시장,품종,포장단위,등급,사이즈,검증일수,단가_정확도,단가_MAE,거래량_정확도,거래량_MAE
창원내서,한라봉,2kg,특,미분류,13일,99.30%,23 원,44.69%,12.8 kg
구리,타이벡감귤,5kg,무등급,미분류,8일,99.26%,26 원,75.94%,479.2 kg
서울가락,온주,8kg,특,특대,5일,99.19%,56 원,53.01%,90.2 kg
구리,감귤(일반),5kg,특,미분류,6일,99.18%,45 원,66.34%,35.9 kg
창원내서,한라봉,12kg,특,미분류,17일,99.12%,31 원,52.31%,9.8 kg
순천,조생귤,5kg,등외,40내,7일,98.72%,3 원,53.48%,3.0 kg
구리,조생귤,5kg,무등급,미분류,9일,98.69%,51 원,76.29%,441.9 kg
서울가락,조생귤,9.6kg,특,미분류,27일,98.64%,121 원,50.21%,157.6 kg
충주,하우스감귤,10kg,특,기타,5일,98.15%,102 원,50.04%,60.0 kg
익산,한라봉,12kg,미분류,미분류,5일,98.07%,67 원,65.79%,4.9 kg


4단계 완료! 실시간 기준일: 2026-11-17 (최대 예측 가능일: 2026-12-17)
5단계 완료! 이어서 6단계를 실행하세요.


Output()